# MCDI502 · Evaluación Sumativa 4 · Taller III
## Integración, limpieza, validación y análisis de ventas con Apache Spark en Databricks
**Grupo 2:** Verónica Durán · Raúl Moya · Daniela Rojas · Manuel Sánchez  
**Fuentes:** `ventas_concepcion.csv`, `ventas_santiago.csv`, `ventas_vina_del_mar.csv`  
**Periodo registrado:** octubre–diciembre de 2024 (según los archivos originales).

**Objetivo.** Integramos las ventas de tres sucursales, corregimos problemas de calidad, verificamos reglas de negocio y obtenemos indicadores mediante Spark SQL. El notebook está diseñado para ejecutarse en un *workspace* de Databricks con una sesión Spark disponible.

**Nota:** Los archivos `ventas_region_norte.csv` y `ventas_region_sur.csv` no forman parte de esta evaluación.

## 0. Preparación del entorno y archivos

**Grupo 2:** Verónica Durán, Raúl Moya, Daniela Rojas y Manuel Sánchez.

El notebook funciona en **VS Code** (pruebas locales) y en **Databricks** (entrega oficial). En VS Code, colocamos los tres archivos CSV en una carpeta `datos` junto al notebook, o en `Sumativa 4 - Taller III/datos`. El código busca ambas ubicaciones y también `Descargas` para facilitar el inicio. **No utiliza contraseñas ni rutas personales fijas.**

En Databricks, cargamos los CSV a un volumen de Unity Catalog o a DBFS y actualizamos el widget `ruta_csv` con la carpeta real. Solo se utilizan los archivos `ventas_concepcion.csv`, `ventas_santiago.csv` y `ventas_vina_del_mar.csv`. Los archivos regionales norte/sur no corresponden a esta evaluación.

> El desarrollo y la ejecución final deben realizarse en Databricks, según las instrucciones de la asignatura.

In [15]:
from pathlib import Path
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

ARCHIVOS_REQUERIDOS = (
    "ventas_vina_del_mar.csv", "ventas_santiago.csv", "ventas_concepcion.csv"
)

# Databricks proporciona dbutils; en VS Code no existe.
try:
    dbutils
    EN_DATABRICKS = True
except NameError:
    EN_DATABRICKS = False

if EN_DATABRICKS:
    try:
        spark
    except NameError:
        spark = SparkSession.builder.getOrCreate()
    dbutils.widgets.text("ruta_csv", "/Volumes/workspace/default/ventas_sumativa4", "Carpeta CSV")
    RUTA_CSV = dbutils.widgets.get("ruta_csv").strip().rstrip("/")
    if not RUTA_CSV:
        raise ValueError("En Databricks indica la carpeta real de los CSV en el widget ruta_csv")
else:
    spark = (SparkSession.builder.master("local[2]")
             .appName("MCDI502_Sumativa4_Grupo2").getOrCreate())

    # Primero se busca en la carpeta actual y sus subcarpetas habituales.
    actual = Path.cwd()
    candidatos = [actual / "datos", actual, actual / "Sumativa 4 - Taller III" / "datos"]
    for padre in list(actual.parents)[:4]:
        candidatos.extend([padre / "datos", padre / "Sumativa 4 - Taller III" / "datos"])
    candidatos.extend([Path.home() / "Downloads", Path.home() / "Descargas"])
    carpeta_encontrada = next(
        (carpeta for carpeta in candidatos
         if all((carpeta / archivo).is_file() for archivo in ARCHIVOS_REQUERIDOS)), None
    )
    if carpeta_encontrada is None:
        raise FileNotFoundError(
            "No se encontraron los tres CSV juntos. Crea la carpeta 'datos' junto al notebook "
            "y copia allí ventas_vina_del_mar.csv, ventas_santiago.csv y ventas_concepcion.csv. "
            f"Carpeta actual: {actual}"
        )
    RUTA_CSV = carpeta_encontrada.resolve().as_posix()

print("Entorno:", "Databricks" if EN_DATABRICKS else "VS Code local")
print("Versión Spark:", spark.version)
print("Ruta CSV:", RUTA_CSV)


Entorno: VS Code local
Versión Spark: 4.0.5
Ruta CSV: C:/Users/eliza/Desktop/ESTUDIOS/MAGÍSTER/3 GESTIÓN DE DATOS Y TECNOLOGÍAS BIG DATA/Sumativa 4 - Taller III/datos


## 1. Carga e integración de datos (ID3.1)
Cargamos cada sucursal por separado, verificamos su estructura y realizamos la unión por **nombre de columna**, evitando depender del orden de los campos. Conservamos inicialmente los precios y fechas como texto para corregirlos de manera controlada.

In [16]:
archivos = {
    "Viña del Mar": "ventas_vina_del_mar.csv",
    "Santiago": "ventas_santiago.csv",
    "Concepción": "ventas_concepcion.csv",
}

def leer_csv(nombre):
    ruta = f"{RUTA_CSV}/{nombre}"
    try:
        df = (spark.read.option("header", True)
              .option("inferSchema", False).option("encoding", "UTF-8").csv(ruta))
        df.limit(1).collect()  # Forzar lectura y detectar errores de ruta.
        columnas = {"id_transaccion", "fecha", "producto_nombre", "categoria",
                    "cantidad", "precio_unitario", "sucursal"}
        faltantes = columnas - set(df.columns)
        if faltantes:
            raise ValueError(f"Faltan columnas en {nombre}: {sorted(faltantes)}")
        return df
    except Exception as e:
        raise RuntimeError(
            f"No se pudo leer {ruta}. Comprueba que los tres CSV estén juntos en la carpeta configurada."
        ) from e

df_vina = leer_csv(archivos["Viña del Mar"])
df_santiago = leer_csv(archivos["Santiago"])
df_concepcion = leer_csv(archivos["Concepción"])

for nombre, df in [("Viña del Mar", df_vina), ("Santiago", df_santiago),
                   ("Concepción", df_concepcion)]:
    print(f"\n=== {nombre} ===")
    df.printSchema()
    df.show(5, truncate=False)



=== Viña del Mar ===
root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- producto_nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- sucursal: string (nullable = true)

+--------------+----------+-----------+-----------------+--------------+--------+---------------+----------+------------+
|id_transaccion|fecha     |id_producto|producto_nombre  |categoria     |cantidad|precio_unitario|id_cliente|sucursal    |
+--------------+----------+-----------+-----------------+--------------+--------+---------------+----------+------------+
|TRX10030      |01/10/2024|PROD889    |Mouse Inalámbrico|Almacenamiento|8       |79990          |CLI1158   |Viña del Mar|
|TRX10065      |2024/10/13|PROD403    |Teclado Mecánico |Almacenamiento|-1      |1              |CLI5

In [17]:
conteos_origen = {"Viña del Mar": df_vina.count(), "Santiago": df_santiago.count(), "Concepción": df_concepcion.count()}
df_ventas_unificado = df_vina.unionByName(df_santiago).unionByName(df_concepcion)
filas_unificadas = df_ventas_unificado.count()
print("Registros por sucursal:", conteos_origen)
print("Suma original:", sum(conteos_origen.values()))
print("Registros unificados:", filas_unificadas)
assert filas_unificadas == sum(conteos_origen.values()), "La unión perdió o duplicó filas"
print("Integridad de unión: OK")

Registros por sucursal: {'Viña del Mar': 105, 'Santiago': 125, 'Concepción': 85}
Suma original: 315
Registros unificados: 315
Integridad de unión: OK


## 2. Diagnóstico y limpieza (ID4.2)
**Decisiones de limpieza:** reemplazamos las categorías vacías por `Sin categoría`, porque eliminar estas ventas produciría pérdida de información económica; eliminamos únicamente duplicados **idénticos en todas las columnas**; normalizamos nombres de productos en minúsculas y espacios; interpretamos explícitamente los formatos mixtos de fecha; retiramos símbolos monetarios de los precios antes de convertirlos a `double`. Conservamos los valores originales en el diagnóstico para auditar el proceso.

In [18]:
def resumen_calidad(df, etiqueta):
    total = df.count()
    nulos = df.select(*[
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).first().asDict()
    duplicados = total - df.dropDuplicates().count()
    print(f"\n=== CALIDAD: {etiqueta} ===")
    print("Filas:", total, "| Duplicados completos:", duplicados)
    print("Nulos / vacíos por columna:", nulos)
    return {"etapa": etiqueta, "filas": total, "duplicados": duplicados, "nulos": nulos}

calidad_antes = resumen_calidad(df_ventas_unificado, "Antes de limpiar")


=== CALIDAD: Antes de limpiar ===
Filas: 315 | Duplicados completos: 15
Nulos / vacíos por columna: {'id_transaccion': 0, 'fecha': 0, 'id_producto': 0, 'producto_nombre': 0, 'categoria': 46, 'cantidad': 0, 'precio_unitario': 0, 'id_cliente': 0, 'sucursal': 0}


In [19]:
# Eliminamos duplicados completos ANTES de transformar, para no confundirlos
# con registros que solo se parecen después de normalizar el texto.
df_sin_duplicados = df_ventas_unificado.dropDuplicates()

categoria_normalizada = F.trim(F.col("categoria"))
precio_texto = F.regexp_replace(F.trim(F.col("precio_unitario")), r"[^0-9,.-]", "")
# En estos tres CSV los precios son enteros (pesos chilenos) con o sin '$'.
# La sustitución de coma decimal se deja explícita por robustez.
precio_normalizado = F.regexp_replace(precio_texto, ",", ".")

# Los CSV combinan yyyy-MM-dd, yyyy/MM/dd, dd/MM/yyyy y dd-MM-yyyy.
# try_to_date no está disponible en todas las versiones; usamos to_date
# después de identificar el patrón, evitando errores de ANSI por formatos mixtos.
fecha_original = F.trim(F.col("fecha"))
fecha_iso = (
    F.when(fecha_original.rlike(r"^\d{4}[-/]\d{2}[-/]\d{2}$"),
           F.regexp_replace(fecha_original, "/", "-"))
     .when(fecha_original.rlike(r"^\d{2}[-/]\d{2}[-/]\d{4}$"),
           F.concat(F.substring(fecha_original, 7, 4), F.lit("-"),
                    F.substring(fecha_original, 4, 2), F.lit("-"),
                    F.substring(fecha_original, 1, 2)))
     .otherwise(F.lit(None).cast("string"))
)

df_limpieza = (df_sin_duplicados
    .withColumn("categoria", F.when(F.col("categoria").isNull() | (categoria_normalizada == ""),
                                     F.lit("Sin categoría")).otherwise(categoria_normalizada))
    .withColumn("producto_nombre", F.lower(F.regexp_replace(F.trim(F.col("producto_nombre")), r"\s+", " ")))
    .withColumn("fecha_estandarizada", fecha_iso)
    .withColumn("fecha", F.expr("try_cast(fecha_estandarizada AS DATE)"))
    .drop("fecha_estandarizada")
    .withColumn("precio_normalizado_tmp", precio_normalizado)
    .withColumn("precio_unitario", F.expr("try_cast(precio_normalizado_tmp AS DOUBLE)"))
    .drop("precio_normalizado_tmp")
)
df_limpieza = df_limpieza.withColumn("cantidad", F.expr("try_cast(cantidad AS INT)"))

print("Después de estandarizar:")
df_limpieza.select("fecha", "producto_nombre", "categoria", "cantidad", "precio_unitario").show(10, truncate=False)

Después de estandarizar:
+----------+---------------------+--------------+--------+---------------+
|fecha     |producto_nombre      |categoria     |cantidad|precio_unitario|
+----------+---------------------+--------------+--------+---------------+
|2024-10-18|ram 16gb             |Redes         |9       |199990.0       |
|2024-11-12|disco ssd 1tb        |Redes         |1       |449990.0       |
|2024-11-29|memoria ram 16gb     |Almacenamiento|2       |199990.0       |
|2024-11-22|laptop gaming        |Almacenamiento|8       |299990.0       |
|2024-10-30|laptop gaming        |Computación   |5       |1.0            |
|2024-12-10|laptop gamer         |Almacenamiento|7       |79990.0        |
|2024-10-19|teclado mecánico     |Audio         |10      |25990.0        |
|2024-12-05|impresora laser      |Redes         |9       |89990.0        |
|2024-12-04|auriculares bluetooth|Almacenamiento|4       |25990.0        |
|2024-11-26|memoria ram 16gb     |Redes         |9       |79990.0        |


**Control de interpretación.** Las fechas se transforman en formato ISO (`yyyy-MM-dd`). El precio se limpia antes de convertirlo; si aparecieran valores no numéricos en otro archivo, debemos investigarlos antes de incorporarlos al análisis.

In [20]:
# Comprobamos que los campos esenciales sean utilizables.
invalidos_formato = df_limpieza.filter(
    F.col("fecha").isNull() | F.col("precio_unitario").isNull() |
    F.col("cantidad").isNull()
)
print("Registros con fecha, precio o cantidad no interpretables:", invalidos_formato.count())
invalidos_formato.show(10, truncate=False)

Registros con fecha, precio o cantidad no interpretables: 0
+--------------+-----+-----------+---------------+---------+--------+---------------+----------+--------+
|id_transaccion|fecha|id_producto|producto_nombre|categoria|cantidad|precio_unitario|id_cliente|sucursal|
+--------------+-----+-----------+---------------+---------+--------+---------------+----------+--------+
+--------------+-----+-----------+---------------+---------+--------+---------------+----------+--------+



## 3. Transformación y validación (ID4.1, ID4.3)
Excluimos cantidades menores o iguales a cero y registros sin fecha o precio válido. También excluimos precios unitarios negativos, ya que no representan una venta convencional. Calculamos `precio_total`, `dia_semana` y verificamos que no existan totales negativos. Los valores atípicos de precio se **marcan**, no se eliminan automáticamente: pueden ser errores de carga o productos genuinamente caros.

In [21]:
df_ventas_filtrado = df_limpieza.filter(
    (F.col("cantidad") > 0) & (F.col("precio_unitario") >= 0) &
    F.col("fecha").isNotNull() & F.col("precio_unitario").isNotNull()
)
filas_excluidas = df_limpieza.count() - df_ventas_filtrado.count()
print("Filas excluidas por reglas de negocio:", filas_excluidas)

df_ventas_transformado = (df_ventas_filtrado
    .withColumn("precio_total", F.round(F.col("cantidad") * F.col("precio_unitario"), 2))
    .withColumn("dia_semana", F.date_format(F.col("fecha"), "EEEE")))

negativos = df_ventas_transformado.filter(F.col("precio_total") < 0).count()
assert negativos == 0, "Se detectaron precios totales negativos"
print("Totales negativos:", negativos, "| Validación: OK")
print("Sucursales distintas:", df_ventas_transformado.select("sucursal").distinct().count())
df_ventas_transformado.select("sucursal").distinct().show(truncate=False)
print("Categorías distintas:", df_ventas_transformado.select("categoria").distinct().count())
df_ventas_transformado.select("categoria").distinct().orderBy("categoria").show(truncate=False)

Filas excluidas por reglas de negocio: 26
Totales negativos: 0 | Validación: OK
Sucursales distintas: 3
+------------+
|sucursal    |
+------------+
|Viña del Mar|
|Santiago    |
|Concepción  |
+------------+

Categorías distintas: 6
+--------------+
|categoria     |
+--------------+
|Almacenamiento|
|Audio         |
|Computación   |
|Periféricos   |
|Redes         |
|Sin categoría |
+--------------+



In [22]:
# IQR: atípico si precio < Q1 - 1.5*IQR o > Q3 + 1.5*IQR.
q1, q3 = df_ventas_transformado.approxQuantile("precio_unitario", [0.25, 0.75], 0.0)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

df_ventas_final = df_ventas_transformado.withColumn(
    "es_outlier_precio",
    (F.col("precio_unitario") < F.lit(limite_inferior)) |
    (F.col("precio_unitario") > F.lit(limite_superior))
)

print(f"Q1={q1:,.2f} | Q3={q3:,.2f} | IQR={iqr:,.2f}")
print(f"Límites IQR: [{limite_inferior:,.2f}, {limite_superior:,.2f}]")
print("Atípicos identificados:", df_ventas_final.filter(F.col("es_outlier_precio")).count())
df_ventas_final.filter(F.col("es_outlier_precio")).select(
    "id_transaccion", "sucursal", "producto_nombre", "precio_unitario", "es_outlier_precio"
).orderBy(F.desc("precio_unitario")).show(20, truncate=False)

Q1=79,990.00 | Q3=299,990.00 | IQR=220,000.00
Límites IQR: [-250,010.00, 629,990.00]
Atípicos identificados: 7
+--------------+------------+---------------------+---------------+-----------------+
|id_transaccion|sucursal    |producto_nombre      |precio_unitario|es_outlier_precio|
+--------------+------------+---------------------+---------------+-----------------+
|TRX20037      |Santiago    |mouse inalambrico    |9999990.0      |true             |
|TRX30009      |Concepción  |impresora láser      |9999990.0      |true             |
|TRX20119      |Santiago    |auriculares bluetooth|9999990.0      |true             |
|TRX20001      |Santiago    |disco ssd 1tb        |9999990.0      |true             |
|TRX30045      |Concepción  |laptop gamer         |9999990.0      |true             |
|TRX20047      |Santiago    |router wifi          |8888888.0      |true             |
|TRX10037      |Viña del Mar|webcam hd            |8888888.0      |true             |
+--------------+------------+

### 3.1 Informe comparativo de calidad
Contrastamos el dataset bruto con el dataset final, contando filas, duplicados completos y valores nulos/vacíos por columna. Las nuevas columnas calculadas solo existen en la etapa final, por lo que la comparación de nulos se centra en las columnas originales.

In [23]:
calidad_despues = resumen_calidad(df_ventas_final, "Después de limpiar y validar")
columnas_origen = df_ventas_unificado.columns
comparacion = [(c, calidad_antes["nulos"].get(c, 0), calidad_despues["nulos"].get(c, 0)) for c in columnas_origen]
df_comparacion_nulos = spark.createDataFrame(comparacion, ["columna", "nulos_antes", "nulos_despues"])
print("Comparación de nulos por columna:")
df_comparacion_nulos.show(truncate=False)
print("Resumen de filas y duplicados:")
spark.createDataFrame([
    ("Antes", calidad_antes["filas"], calidad_antes["duplicados"]),
    ("Después", calidad_despues["filas"], calidad_despues["duplicados"])
], ["etapa", "filas", "duplicados_completos"]).show(truncate=False)
print("Filas eliminadas en total:", calidad_antes["filas"] - calidad_despues["filas"])
assert calidad_despues["duplicados"] == 0, "Persisten duplicados completos"


=== CALIDAD: Después de limpiar y validar ===
Filas: 274 | Duplicados completos: 0
Nulos / vacíos por columna: {'id_transaccion': 0, 'fecha': 0, 'id_producto': 0, 'producto_nombre': 0, 'categoria': 0, 'cantidad': 0, 'precio_unitario': 0, 'id_cliente': 0, 'sucursal': 0, 'precio_total': 0, 'dia_semana': 0, 'es_outlier_precio': 0}
Comparación de nulos por columna:
+---------------+-----------+-------------+
|columna        |nulos_antes|nulos_despues|
+---------------+-----------+-------------+
|id_transaccion |0          |0            |
|fecha          |0          |0            |
|id_producto    |0          |0            |
|producto_nombre|0          |0            |
|categoria      |46         |0            |
|cantidad       |0          |0            |
|precio_unitario|0          |0            |
|id_cliente     |0          |0            |
|sucursal       |0          |0            |
+---------------+-----------+-------------+

Resumen de filas y duplicados:
+-------+-----+----------------

## 4. Consultas con Spark SQL (ID3.3)
Creamos la vista temporal `ventas_limpias` desde el DataFrame final. Las tres consultas siguientes corresponden exactamente a las solicitadas en la evaluación.

In [24]:
df_ventas_final.createOrReplaceTempView("ventas_limpias")
print("Vista SQL creada: ventas_limpias")

Vista SQL creada: ventas_limpias


### Consulta 1: las 10 ventas con mayor importe
**Interpretación:** identificamos las transacciones de mayor valor. Es importante revisar la columna `es_outlier_precio` antes de tomar decisiones sobre estas ventas.

In [25]:
consulta_1 = spark.sql("""
SELECT id_transaccion, fecha, sucursal, producto_nombre, precio_total
FROM ventas_limpias
ORDER BY precio_total DESC, id_transaccion
LIMIT 10
""")
consulta_1.show(10, truncate=False)

+--------------+----------+------------+---------------------+------------+
|id_transaccion|fecha     |sucursal    |producto_nombre      |precio_total|
+--------------+----------+------------+---------------------+------------+
|TRX20047      |2024-12-07|Santiago    |router wifi          |7.9999992E7 |
|TRX30045      |2024-10-01|Concepción  |laptop gamer         |7.999992E7  |
|TRX20001      |2024-11-23|Santiago    |disco ssd 1tb        |6.999993E7  |
|TRX20119      |2024-12-19|Santiago    |auriculares bluetooth|6.999993E7  |
|TRX10037      |2024-11-10|Viña del Mar|webcam hd            |3.5555552E7 |
|TRX20037      |2024-12-16|Santiago    |mouse inalambrico    |2.999997E7  |
|TRX30009      |2024-10-20|Concepción  |impresora láser      |1.999998E7  |
|TRX10074      |2024-11-25|Viña del Mar|teclado mecanico     |4499900.0   |
|TRX20014      |2024-10-15|Santiago    |teclado mecánico     |4499900.0   |
|TRX20063      |2024-10-17|Santiago    |laptop gamer         |4049910.0   |
+-----------

### Consulta 2: monto total y cantidad de transacciones por sucursal
**Interpretación:** comparamos el desempeño comercial entre Concepción, Santiago y Viña del Mar. Se cuentan filas de transacciones después de limpiar y validar.

In [26]:
consulta_2 = spark.sql("""
SELECT sucursal,
       ROUND(SUM(precio_total), 2) AS ventas_totales,
       COUNT(*) AS cantidad_transacciones
FROM ventas_limpias
GROUP BY sucursal
ORDER BY ventas_totales DESC
""")
consulta_2.show(truncate=False)

+------------+--------------+----------------------+
|sucursal    |ventas_totales|cantidad_transacciones|
+------------+--------------+----------------------+
|Santiago    |3.58320486E8  |112                   |
|Concepción  |1.5611041E8   |70                    |
|Viña del Mar|1.24594447E8  |92                    |
+------------+--------------+----------------------+



### Consulta 3: precio promedio, mínimo y máximo por categoría
**Interpretación:** examinamos la dispersión de precios entre categorías. `Sin categoría` representa valores imputados, no una clasificación comercial confirmada.

In [27]:
consulta_3 = spark.sql("""
SELECT categoria,
       ROUND(AVG(precio_unitario), 2) AS precio_promedio,
       MIN(precio_unitario) AS precio_minimo,
       MAX(precio_unitario) AS precio_maximo
FROM ventas_limpias
GROUP BY categoria
ORDER BY categoria
""")
consulta_3.show(truncate=False)

+--------------+---------------+-------------+-------------+
|categoria     |precio_promedio|precio_minimo|precio_maximo|
+--------------+---------------+-------------+-------------+
|Almacenamiento|200602.24      |25990.0      |449990.0     |
|Audio         |549928.4       |1.0          |9999990.0    |
|Computación   |144365.23      |1.0          |449990.0     |
|Periféricos   |177102.44      |50.0         |449990.0     |
|Redes         |392705.98      |1.0          |9999990.0    |
|Sin categoría |1148463.95     |50.0         |9999990.0    |
+--------------+---------------+-------------+-------------+



## 5. Conclusiones y verificación de entrega
- La unión por nombre de columna conserva la totalidad de las filas originales antes de depurar.
- Imputamos `Sin categoría` para mantener ventas sin clasificación; eliminamos solo duplicados completos y registros que no cumplen reglas de negocio.
- La estandarización de fechas, precios y nombres permite análisis coherentes.
- Marcamos los valores atípicos mediante IQR y **no** los descartamos sin una revisión adicional.
- Los resultados de Spark SQL permiten comparar sucursales y categorías, pero deben interpretarse considerando los datos imputados y los outliers.

**Verificación final:** ejecutar todas las celdas en Databricks, revisar los conteos, confirmar que las tres consultas producen resultados y exportar el notebook con nombre `mcdi502_s4_grupo2`. Comprimir el notebook en un `.zip` para entregar en Canvas. No se solicita video en esta Sumativa 4.

**Nota metodológica:** los valores atípicos se identifican y conservan marcados; no se eliminan automáticamente porque podrían corresponder a ventas legítimas. Las fechas y precios no interpretables se excluyen y contabilizan. La vista SQL se genera sobre el conjunto depurado.

In [28]:
print("=== CONTROL FINAL ===")
print("Archivos de origen:", len(archivos))
print("Filas iniciales:", filas_unificadas)
print("Filas finales:", df_ventas_final.count())
print("Duplicados finales:", df_ventas_final.count() - df_ventas_final.dropDuplicates().count())
print("Precios totales negativos:", df_ventas_final.filter(F.col("precio_total") < 0).count())
print("Atípicos marcados:", df_ventas_final.filter(F.col("es_outlier_precio")).count())
print("Consultas SQL: 3 de 3")
assert df_ventas_final.select("sucursal").distinct().count() == 3
print("Validación final completada")

=== CONTROL FINAL ===
Archivos de origen: 3
Filas iniciales: 315
Filas finales: 274
Duplicados finales: 0
Precios totales negativos: 0
Atípicos marcados: 7
Consultas SQL: 3 de 3
Validación final completada
